# 10 · End-to-End Project

> **The capstone · Modules 1–8 · A production-shaped Beam job, end to end**

Every course so far has taught one thing at a time. This notebook puts
the pieces together and runs the **course's actual project code** — not a
simplified imitation of it.

You will:

- run the batch pipeline and assert the exact revenue totals for four
  regions;
- run the streaming pipeline over the same generated data and read the
  windowed counts;
- **cross-validate the Beam result against the pure reference
  implementation** — the single most important moment in the notebook;
- mirror the project's `pytest` suite with `TestPipeline` and
  `assert_that`;
- and finish with what changes — and what does not — when the same graph
  is deployed to Dataflow.

Everything below runs **locally, with no GCP account, no credentials and
no bill**. The Dataflow section at the end is prose, plus one code block
that stays inert unless you export `PROJECT_ID` and `GCS_BUCKET`.

---

## 1. Setup

The setup cell puts the course's `project/` directory on `sys.path` so
that `batch_pipeline`, `streaming_pipeline` and `transforms` can be
imported as ordinary modules, generates the sample data if it is not
there yet, and prepares a scratch directory for the outputs.

Two conventions worth noting:

- the **runner is pinned to `BundleBasedDirectRunner`**, so nothing here
  reaches for the network or for a credential;
- the sample data is generated on demand, so the notebook works in a
  fresh checkout with nothing but `apache-beam` installed.

In [ ]:
# --- Setup -----------------------------------------------------------------
import csv
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = COURSE_ROOT.parent

PROJECT_DIR = COURSE_ROOT / "project"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

if not (COURSE_ROOT / "sample_data" / "orders.csv").exists():
    subprocess.run(
        [sys.executable, str(COURSE_ROOT / "resources" / "sample_data.py"),
         "--output-dir", str(COURSE_ROOT / "sample_data")],
        check=True,
    )

import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

from batch_pipeline import build_pipeline, run as run_batch
from streaming_pipeline import build_streaming_pipeline
from transforms import build_revenue_summary, parse_event_payload

LOCAL_OPTIONS = PipelineOptions(["--runner", "BundleBasedDirectRunner"])

SCRATCH = COURSE_ROOT / "out" / "nb10"
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)
SCRATCH.mkdir(parents=True, exist_ok=True)

ORDERS_CSV = str(COURSE_ROOT / "sample_data" / "orders.csv")
EVENTS_NDJSON = str(COURSE_ROOT / "sample_data" / "events.ndjson")

print("beam       :", beam.__version__)
print("course root:", COURSE_ROOT)
print("scratch    :", SCRATCH)
print()
print("The project modules we are about to import:")
for name in ("batch_pipeline.py", "streaming_pipeline.py", "transforms.py"):
    print(f"   {name}")

## 2. The batch pipeline

The batch pipeline is a five-step graph:

```text
ReadFromText
    -> ParseOrderFn (ParDo)      parse, validate, convert currency to EUR
    -> Filter                     keep only the requested statuses
    -> KeyByRegion                (region, order)
    -> CombinePerKey              RevenueByRegionFn
    -> WriteToText                a CSV summary
```

`build_pipeline` takes a `beam.Pipeline` and adds its transforms to it.
That signature is the important design decision: **the function that
builds the graph and the function that runs the pipeline are separate**,
so the same graph can be driven by the command line, by a test, or by
this notebook.

We will call it three ways in this notebook:

1. through `beam.Pipeline()` directly, to show the graph being assembled;
2. through the `run()` entry point, to show the production path;
3. through `TestPipeline`, in the test section.

### Reading the output

`WriteToText` writes a **file**, not a row. The summary is one CSV file
whose name is the output prefix, and the first line is the header. That
is worth internalising: a Beam sink is a *file sink*, and reading its
output back is an ordinary file read, not a Beam operation.

The file is named exactly the prefix because the project sets
`shard_name_template=""`. If you change that, the output becomes a
directory of shard files and the read-back has to glob.

### Asserting the totals

A number printed on screen is not a check. A number **asserted** is.

The expected totals are not arbitrary: they are what the course's case
study reports, and they are what a correct implementation of the rules
must produce. 1,245 of the 5,000 orders have the `complete` status, and
after the currency conversion the revenue sums to 450,856.51 EUR.

The assertion below is the contract. If a change to the pipeline breaks
the currency table, or moves the status filter, or changes how the
combiner accumulates, this line fails — loudly, and at the point of the
change rather than three weeks later in a dashboard.

```python
assert batch_result == EXPECTED
```

That is a strong assertion: it checks the count *and* the revenue, for
*every* region, and it fails if a region appears that should not. It
also fails on floating point noise, which is why the expected values are
written to two decimal places and the comparison is done on rounded
values.

In [ ]:
# --- Build and run the batch pipeline --------------------------------------
#
# `build_pipeline` takes the pipeline and the paths. It writes the
# summary CSV and returns None -- the result is on disk, not in memory.

BATCH_OUT = str(SCRATCH / "summary")

with beam.Pipeline(options=LOCAL_OPTIONS) as pipeline:
    build_pipeline(
        pipeline,
        input_path=ORDERS_CSV,
        output_path=BATCH_OUT,
        statuses=("complete",),
    )


def read_summary_csv(path):
    # Read a summary CSV into {region: (count, revenue)}.
    #
    # The sink writes a single file whose name is the output prefix, and
    # the first line is the header.
    candidate = Path(path)
    files = [candidate] if candidate.is_file() else sorted(candidate.glob("*"))

    rows = {}
    for file_path in files:
        text = file_path.read_text(encoding="utf-8")
        for line in text.strip().splitlines():
            if not line or line.startswith("region,"):
                continue
            region, count, revenue = line.split(",")
            rows[region] = (int(count), float(revenue))
    return rows


batch_result = read_summary_csv(BATCH_OUT)

print(f"{'region':<8} {'order_count':>12} {'revenue_eur':>14}")
print("-" * 38)
for region in sorted(batch_result):
    count, revenue = batch_result[region]
    print(f"{region:<8} {count:>12} {revenue:>14.2f}")

print()
print("The expected totals, from the course's case study:")
EXPECTED = {
    "AMER": (318, 113592.19),
    "APAC": (285, 108194.92),
    "EMEA": (322, 118423.22),
    "LATAM": (320, 110646.18),
}
for region, (count, revenue) in sorted(EXPECTED.items()):
    print(f"   {region}: {count} orders, {revenue:.2f} EUR")


# --- Assert the exact totals ----------------------------------------------
#
# Compare on rounded values: the combiner accumulates floats, and the
# exact binary representation of 450856.50999999994 is not the same
# object as 450856.51.

rounded = {
    region: (count, round(revenue, 2))
    for region, (count, revenue) in batch_result.items()
}

print("region    beam (count, revenue)      expected (count, revenue)   match")
print("-" * 76)
for region in sorted(EXPECTED):
    got = rounded.get(region)
    want = EXPECTED[region]
    ok = got == want
    print(f"{region:<9} {str(got):<24} {str(want):<24} {'OK' if ok else 'MISMATCH'}")

print()
assert rounded == EXPECTED, f"batch totals differ: {rounded} != {EXPECTED}"
print("The batch pipeline produces the expected totals for all four regions.")

total_orders = sum(count for count, _ in batch_result.values())
total_revenue = sum(revenue for _, revenue in batch_result.values())
print()
print(f"total orders  : {total_orders}")
print(f"total revenue : {total_revenue:.2f} EUR")
assert total_orders == 1245
assert abs(total_revenue - 450856.51) < 0.01
print("and they sum to the case study's 1,245 orders and 450,856.51 EUR.")

## 3. The streaming pipeline

The streaming pipeline is the same idea with time in it:

```text
read -> parse -> window -> key -> count + sum -> CoGroupByKey -> format -> write
```

The differences that matter:

| | Batch | Streaming |
|---|---|---|
| Input | a bounded CSV file | an unbounded event source (here: a bounded NDJSON file) |
| Grouping | by region | by region, **within a window** |
| Trigger | none — the whole file is one group | `AfterWatermark` by default |
| Lateness | not a concept | how long a late event may still update a fired window |

The sample data spans a single day, so a **24-hour window** produces one
window per region and the output is directly comparable to the batch
numbers. A one-hour window produces 24 windows per region, and the
per-window counts sum to the same totals — that is the property worth
showing.

### Why this is still a streaming pipeline

The input here is a **bounded** file, so the pipeline terminates. But
every transform in the graph is the same streaming transform it would be
against Pub/Sub: `WindowInto`, `AfterWatermark`, `CoGroupByKey`. Replace
the source with `ReadFromPubSub` and the graph does not change at all.
That is the point of testing streaming logic against a bounded source:
you get a deterministic, terminating run of the exact code that will
face a real stream.

In [ ]:
# --- Run the streaming pipeline over the bounded event file -----------------
#
# The project resolves the runner from `beam_options.build_options`, so
# we call `build_streaming_pipeline` directly with our own options and
# keep full control of the window width.
#
# We use a ONE-HOUR window, not a 24-hour one. The sample data spans a
# single day, so a one-hour window should produce 24 windows per
# region and the windowing is genuinely exercised -- which is the whole
# point of running the streaming pipeline at all.

WINDOW_SECONDS = 3600
STREAM_OUT = str(SCRATCH / "stream")

with beam.Pipeline(options=LOCAL_OPTIONS) as pipeline:
    events = pipeline | "ReadEvents" >> beam.io.ReadFromText(EVENTS_NDJSON)
    build_streaming_pipeline(
        pipeline,
        events=events,
        output_path=STREAM_OUT,
        window_seconds=WINDOW_SECONDS,
        allowed_lateness_seconds=300,
        trigger_count=0,           # 0 = the safe AfterWatermark trigger
        options=LOCAL_OPTIONS,
    )

stream_text = Path(STREAM_OUT).read_text(encoding="utf-8").strip().splitlines()
print("the rows the pipeline wrote:")
for line in stream_text:
    print("   ", line)
print()

# The sink writes one row per (window, region). Sum them to get the
# per-region totals.
stream_counts = {}
for line in stream_text:
    if not line or line.startswith("region,"):
        continue
    region, count, value = line.split(",")
    entry = stream_counts.setdefault(region, [0, 0.0])
    entry[0] += int(count)
    entry[1] += float(value)

print(f"{'region':<8} {'event_count':>12} {'total_value':>14}")
print("-" * 38)
for region in sorted(stream_counts):
    count, value = stream_counts[region]
    print(f"{region:<8} {count:>12} {value:>14.2f}")

print()
total_events = sum(count for count, _ in stream_counts.values())
print(f"total events : {total_events}")
assert total_events == 20000
print("which is every event the generator wrote.")
print()
print("But look at the shape of the output: FOUR rows, one per region.")
print("With a one-hour window over a whole day of data, there should be")
print("24 windows per region -- 96 rows. There are four.")
print()
print("That is not a bug in the windowing. It is a bug in the TIMESTAMPS.")

## 4. Cross-validation — the single most important moment

The streaming pipeline has just produced a number. How do we know it is
right?

Not by looking at it. By computing the same thing a **second, completely
different way** and requiring the two to agree.

`project/transforms.py` contains `build_revenue_summary`: a pure-Python
loop that implements the same rules with no Beam at all.

```python
def build_revenue_summary(rows, *, statuses=("complete",)):
    ...
```

Two independent implementations of the same specification agreeing is
strong evidence. One implementation agreeing with itself is not. This is
the difference between a pipeline that *runs* and a pipeline that is
*correct*.

The comparison is only meaningful because the two implementations share
almost no code:

| | Beam pipeline | Pure reference |
|---|---|---|
| currency conversion | `normalise_currency` inside `RevenueByRegionFn` | `normalise_currency` inside `build_revenue_summary` |
| validation | `is_valid_order` in `ParseOrderFn` | `is_valid_order` in the loop |
| aggregation | `CombinePerKey` + a `CombineFn` accumulator | a plain `dict` |
| ordering | none guaranteed | none needed |

They share the *rules*. They do not share the *mechanism*. That is the
point: a shared mechanism would hide a shared bug.

### The batch cross-check

`build_revenue_summary` takes order dicts, so we parse the CSV with
`csv.DictReader` and hand the rows over. The two paths differ in how
they read the file, how they filter, and how they aggregate.

### The streaming cross-check

The same argument applies to the streaming pipeline, and it is a little
sharper, because windowing is easy to get wrong in ways that produce
*plausible* numbers.

The reference for the streaming side is not a loop over the same file —
it is a loop that groups by **window index**, using the project's own
`window_key` helper, and then sums. If the Beam pipeline's windowing is
wrong, the two disagree.

We check the **totals** per region against the reference, and we check
the **number of (window, region) groups**. The reference groups by
`(window index, region)` with a plain dict, so the window boundaries
are visible in the reference output rather than hidden inside a trigger.
A pipeline that got the boundaries wrong can still produce the right
totals — and then be completely wrong for every downstream consumer that
cares about *when* something happened.

### The timestamp gap

The cross-check is what exposes a real gap in the course's streaming
pipeline, and it is worth walking through slowly, because the failure
mode is invisible without a second implementation.

`build_streaming_pipeline` windows the events, but **nothing in the
graph sets each element's Beam timestamp from the event's own
`event_ts`**. The project ships an `event_timestamp` helper that does
exactly that, and the docstring explains why it matters — but the
pipeline never calls it.

What Beam does instead is use **arrival time**: every element read from
the file gets roughly the same timestamp, so all 20,000 events land in
one global window. The output is four rows, one per region, and the
per-region totals are **correct** — which is exactly why the bug is
dangerous. Nothing fails. The dashboard looks right. The per-hour
numbers simply do not exist.

The fix is one transform: attach the event's `event_ts` to the
element's timestamp before the `WindowInto`. In Beam that is
`apache_beam.transforms.window.TimestampedValue(value, timestamp)` —
yield one and the timestamp rides alongside the (unmodified) value, all
the way into the windowing. The next cell builds that version in the
notebook, and the cross-check then passes window by window.

In [ ]:
# --- The pure reference implementation -------------------------------------
#
# No Beam here at all: a loop, a dict, and the same rules.

with open(ORDERS_CSV, encoding="utf-8") as fh:
    reference_orders = list(csv.DictReader(fh))

for row in reference_orders:
    row["amount"] = float(row["amount"])

reference = build_revenue_summary(reference_orders, statuses=("complete",))

print("region    beam count   ref count    beam revenue    ref revenue   match")
print("-" * 78)

all_match = True
for region in sorted(EXPECTED):
    beam_count, beam_revenue = batch_result[region]
    ref = reference[region]
    ref_count = int(ref["order_count"])
    ref_revenue = round(float(ref["revenue_eur"]), 2)
    match = (beam_count == ref_count) and (abs(beam_revenue - ref_revenue) < 0.01)
    all_match = all_match and match
    print(f"{region:<9} {beam_count:>10} {ref_count:>11} "
          f"{beam_revenue:>14.2f} {ref_revenue:>14.2f}   {'OK' if match else 'MISMATCH'}")

print()
assert all_match, "the Beam pipeline and the reference implementation disagree"
assert set(reference) == set(batch_result), "the two disagree about which regions exist"
print("The Beam pipeline and the pure reference agree on every region.")
print()
print("This is the assertion that makes the pipeline trustworthy. Without")
print("it, a green run only tells you the pipeline did not crash.")


# --- Cross-validate the streaming pipeline window by window ---------------
import collections

from transforms import window_key

with open(EVENTS_NDJSON, encoding="utf-8") as fh:
    raw_events = [json.loads(line) for line in fh if line.strip()]

# The pure reference: group by (window index, region) with a dict.
#
# `window_key` computes the window index the SAME way Beam does -- by
# dividing the event timestamp by the window width -- but with plain
# integer arithmetic. No Beam, no trigger, no watermark.
reference_windows = collections.defaultdict(lambda: {"count": 0, "value": 0.0})
for raw in raw_events:
    event = parse_event_payload(json.dumps(raw).encode("utf-8"))
    index = window_key(event, WINDOW_SECONDS)
    bucket = reference_windows[(index, event["region"])]
    bucket["count"] += 1
    bucket["value"] += float(event["value"])

window_indices = sorted({index for index, _ in reference_windows})
print(f"the reference found {len(reference_windows)} (window, region) groups")
print(f"across {len(window_indices)} hourly windows")
print()


# --- The same graph, but with the event's own timestamp --------------------
#
# One extra transform: move the event's `event_ts` onto the element's
# Beam timestamp, so `WindowInto` groups by WHEN THE EVENT HAPPENED
# rather than by when the file was read.
#
# We use the project's own `event_timestamp` helper, which is exactly
# what it was written for.
#
# The API that does the work is `TimestampedValue(value, timestamp)`:
# yield one of those and Beam attaches the timestamp to the element. The
# value itself is unchanged -- we still hand the pipeline the raw bytes,
# because `build_streaming_pipeline` parses them itself.
#
# (`apache_beam.window` is an alias for the same module; the canonical
# import path is `apache_beam.transforms.window`.)

from apache_beam.transforms.window import TimestampedValue

from streaming_pipeline import event_timestamp

EVENT_TIME_OUT = str(SCRATCH / "stream_event_time")


class StampEventTimeFn(beam.DoFn):
    # Attach the event's own timestamp to the raw payload.
    #
    # Note what is NOT happening here: the value is not modified. The
    # timestamp rides alongside it, and every downstream transform keeps
    # it until `WindowInto` uses it.
    def process(self, payload):
        event = parse_event_payload(payload)
        yield TimestampedValue(payload, event_timestamp(event))


with beam.Pipeline(options=LOCAL_OPTIONS) as pipeline:
    raw = pipeline | "ReadEvents" >> beam.io.ReadFromText(EVENTS_NDJSON)
    stamped = raw | "WithEventTimestamps" >> beam.ParDo(StampEventTimeFn())
    build_streaming_pipeline(
        pipeline,
        events=stamped,
        output_path=EVENT_TIME_OUT,
        window_seconds=WINDOW_SECONDS,
        allowed_lateness_seconds=300,
        trigger_count=0,
        options=LOCAL_OPTIONS,
    )

event_time_text = Path(EVENT_TIME_OUT).read_text(encoding="utf-8").strip().splitlines()
event_time_rows = [line for line in event_time_text
                   if line and not line.startswith("region,")]

print("the same pipeline, with the event's own timestamp:")
print(f"   rows written : {len(event_time_rows)}")
print(f"   the reference found {len(reference_windows)} (window, region) groups")
print()

# Now the totals.
stream_counts = {}
for line in event_time_rows:
    region, count, value = line.split(",")
    entry = stream_counts.setdefault(region, [0, 0.0])
    entry[0] += int(count)
    entry[1] += float(value)

reference_totals = {}
for (index, region), bucket in reference_windows.items():
    entry = reference_totals.setdefault(region, {"count": 0, "value": 0.0})
    entry["count"] += bucket["count"]
    entry["value"] += bucket["value"]

print(f"{'region':<8} {'beam count':>12} {'ref count':>11} "
      f"{'beam value':>13} {'ref value':>12}   match")
print("-" * 78)

stream_match = True
for region in sorted(reference_totals):
    beam_count, beam_value = stream_counts[region]
    ref = reference_totals[region]
    ref_count = ref["count"]
    ref_value = round(ref["value"], 2)
    match = (beam_count == ref_count) and (abs(beam_value - ref_value) < 0.01)
    stream_match = stream_match and match
    print(f"{region:<8} {beam_count:>12} {ref_count:>11} "
          f"{beam_value:>13.2f} {ref_value:>12.2f}   {'OK' if match else 'MISMATCH'}")

print()
assert stream_match, "the streaming pipeline and the reference disagree"
print("The streaming pipeline agrees with the reference on every region.")
print()

# The per-window check.
#
# The sink does not write the window index, so we cannot join the two
# side by side directly. What we CAN check is that the pipeline emitted
# one row per (window, region) group -- the same number of rows the
# reference found. If the windowing were wrong, the two counts would
# differ: a too-narrow window would produce more groups, and a
# too-wide one fewer.
print("the window structure:")
print(f"   rows the pipeline wrote      : {len(event_time_rows)}")
print(f"   groups the reference found   : {len(reference_windows)}")
print(f"   hourly windows in the data   : {len(window_indices)}")
print()
assert len(event_time_rows) == len(reference_windows), (
    f"the pipeline emitted {len(event_time_rows)} rows but the reference "
    f"found {len(reference_windows)} (window, region) groups -- the "
    f"windowing disagrees"
)
print("The pipeline emitted exactly one row per (window, region) group,")
print("so its window boundaries are the same as the reference's.")
print()
print("Note what the reference does NOT do: it never mentions Beam, never")
print("opens a window, never fires a trigger. It just buckets by a")
print("computed window index. That is why the agreement is meaningful.")

## 5. Idempotency — running it twice

A batch pipeline that cannot be run twice is a batch pipeline that cannot
be re-run after a fix, re-run after a partial failure, or re-run to
backfill a period.

The project's pipeline is idempotent because **nothing in the logic
reads the wall clock** and **the output path is an input, not a
constant**. The only timestamp in the output comes from the data.

So: run it again, to a *different* path, and require the same numbers.

In [ ]:
# --- Run the batch pipeline a second time ---------------------------------
#
# `run` is the production entry point: it parses arguments, resolves the
# runner, and builds the graph. We call it with an explicit local
# runner so no credentials are needed.

RERUN_OUT = str(SCRATCH / "summary_rerun")

exit_code = run_batch([
    "--runner", "BundleBasedDirectRunner",
    "--input", ORDERS_CSV,
    "--output", RERUN_OUT,
    "--statuses", "complete",
])
print("run() exit code:", exit_code)
assert exit_code == 0

rerun_result = read_summary_csv(RERUN_OUT)

print()
print(f"{'region':<8} {'first run':>22} {'second run':>22}   match")
print("-" * 66)
for region in sorted(batch_result):
    first = batch_result[region]
    second = rerun_result[region]
    match = first == second
    print(f"{region:<8} {str(first):>22} {str(second):>22}   {'OK' if match else 'MISMATCH'}")

print()
assert rerun_result == batch_result, "the two runs disagree"
print("The two runs agree exactly.")
print()
print("The output paths differ, so the second run cannot overwrite the")
print("first — which is what makes a backfill safe:")
print("   first :", BATCH_OUT)
print("   second:", RERUN_OUT)

## 6. The test suite, inside the notebook

The project's `pytest` suite lives in `project/tests/`. It is the right
place for these assertions — they run on every commit. But the two
mechanisms it uses are worth seeing in the notebook, because they are
what most Beam tests are made of.

`beam.testing.TestPipeline` is a `Pipeline` that knows how to check
itself. `assert_that` attaches a matcher to a PCollection, and the
pipeline fails if the match fails.

```python
with TestPipeline() as p:
    result = p | ...
    assert_that(result, equal_to([...]))
```

Two details cost everyone an afternoon the first time:

1. **With a windowing transform, the elements arrive as
   `WindowedValue` objects, not bare values.** Comparing them directly
   against plain tuples fails, because a `WindowedValue` also carries a
   window. Pass `use_global_window=False` and unwrap the values, or
   compare the whole `WindowedValue`.

2. **`TestPipeline` replaces the runner.** It runs the graph in a
   special mode so that the assertions can see the output. It is not
   the Direct Runner, and it is not a mock — the transforms really run.

The project uses exactly this pattern, and the helper it writes for the
windowed case is worth copying:

```python
def _windowed_values(expected):
    def _compare(actual):
        unwrapped = [item.value if hasattr(item, "value") else item
                     for item in actual]
        return sorted(unwrapped, key=repr) == sorted(expected, key=repr)
    return _compare
```

### The windowed test

The windowing test is the one that needs the unwrapping helper, and it
is the one that catches the class of bug that only appears in
production: a window that groups the wrong events together.

```text
10:05, 10:55   ->  the 10:00 hour window  ->  one group of 2
11:05          ->  the 11:00 hour window  ->  one group of 1
```

In [ ]:
# --- TestPipeline and assert_that -----------------------------------------
from apache_beam.testing.test_pipeline import TestPipeline
from apache_beam.testing.util import assert_that, equal_to

from transforms import ParseOrderFn, RevenueByRegionFn, to_summary_row

ORDER_LINES = [
    "o1,c1,2026-01-15T10:00:00+00:00,EMEA,web,EUR,complete,100.00,p1",
    "o2,c2,2026-01-15T11:00:00+00:00,EMEA,web,USD,complete,200.00,p2",
    "o3,c3,2026-01-15T12:00:00+00:00,AMER,web,EUR,complete,50.00,p3",
    "o4,c4,2026-01-15T13:00:00+00:00,AMER,web,EUR,cancelled,999.00,p4",
    "o5,c5,2026-01-15T14:00:00+00:00,EMEA,web,GBP,complete,100.00,p5",
]


def _orders(lines, pipeline):
    # Build (region, order) pairs from raw CSV lines.
    return (
        pipeline
        | "Create" >> beam.Create(lines)
        | "Parse" >> beam.ParDo(ParseOrderFn())
        | "KeyByRegion" >> beam.Map(lambda order: (order["region"], order))
    )


# The combiner test: revenue is aggregated per region.
#
# NOTE: `RevenueByRegionFn` does no status filtering of its own -- it
# counts every order it is given. The status filter lives UPSTREAM in
# the pipeline. So here the cancelled order is included, and AMER
# counts 2.
with TestPipeline(options=LOCAL_OPTIONS) as pipeline:
    result = (
        _orders(ORDER_LINES, pipeline)
        | "Revenue" >> beam.CombinePerKey(RevenueByRegionFn())
        | "ToRows" >> beam.Map(to_summary_row)
    )
    assert_that(
        result,
        equal_to([
            # EMEA = EUR 100 + USD 200 x 0.92 (184) + GBP 100 x 1.17 (117)
            {"region": "EMEA", "order_count": 3, "revenue_eur": 401.0},
            # AMER = EUR 50 + cancelled EUR 999
            {"region": "AMER", "order_count": 2, "revenue_eur": 1049.0},
        ]),
    )

print("The combiner test passed: EMEA 3 orders / 401.00 EUR, AMER 2 / 1049.00.")
print()
print("Note the currency arithmetic in the expected value: the test does")
print("not just check the sum, it checks the CONVERSION. A pipeline that")
print("added raw amounts would give 400.00 here, not 401.00.")
print()
print("Note also that the cancelled order is INCLUDED. The combiner does")
print("not filter -- the status filter is a separate step upstream. That")
print("separation is what makes the combiner reusable, and it is also")
print("why the filter has to be tested separately.")


# --- A windowed test, with the unwrapping --------------------------------
from apache_beam.transforms.window import FixedWindows

from transforms import parse_timestamp


def _to_epoch(timestamp):
    return float(parse_timestamp(timestamp).timestamp())


def _windowed_values(expected):
    # Compare the VALUES, not the windows.
    #
    # With `use_global_window=False` the elements arrive as
    # `WindowedValue` objects. Comparing them directly against plain
    # tuples fails, because each one also carries a window.

    def _compare(actual):
        unwrapped = [
            item.value if hasattr(item, "value") else item for item in actual
        ]
        return sorted(unwrapped, key=repr) == sorted(expected, key=repr)

    return _compare


EVENTS = [
    ("2026-01-15T10:05:00+00:00", 10.0),
    ("2026-01-15T10:55:00+00:00", 20.0),
    ("2026-01-15T11:05:00+00:00", 30.0),
]

with TestPipeline(options=LOCAL_OPTIONS) as pipeline:
    result = (
        pipeline
        | "Create" >> beam.Create(EVENTS)
        | "WithTimestamps" >> beam.Map(
            lambda pair: (beam.window.Timestamp(_to_epoch(pair[0])), pair[1])
        )
        | "Window" >> beam.WindowInto(FixedWindows(3600))
        | "KeyByRegion" >> beam.Map(lambda pair: ("all", pair[1]))
        | "Count" >> beam.combiners.Count.PerKey()
    )
    assert_that(
        result,
        _windowed_values([("all", 2), ("all", 1)]),
        use_global_window=False,
    )

print("The windowing test passed: two events in the 10:00 window, one in")
print("the 11:00 window, so two groups.")
print()
print("If the window width were wrong -- say FixedWindows(7200) -- the")
print("three events would land in ONE group and this assertion would")
print("fail. That is the bug this test exists to catch.")

## 7. How this deploys to Dataflow

Everything above ran on the local runner. The deployment story is
short, and that is not an accident — it is the payoff of the design.

### What changes

Four things, and only four:

| Change | Local | Dataflow |
|---|---|---|
| runner | `--runner BundleBasedDirectRunner` | `--runner DataflowRunner` |
| input path | `./sample_data/orders.csv` | `gs://my-bucket/orders.csv` |
| output path | `./out/summary` | `gs://my-bucket/summary` |
| staging and temp | not needed | `gs://my-bucket/dataflow/staging`, `.../temp` |

Plus a `--project` and a `--region`, and a service account that can
create and run Dataflow jobs.

The command is the same script with different flags:

```bash
python project/batch_pipeline.py \
    --runner DataflowRunner \
    --project my-project \
    --region us-central1 \
    --input gs://my-bucket/orders.csv \
    --output gs://my-bucket/summary \
    --staging_location gs://my-bucket/dataflow/staging \
    --temp_location gs://my-bucket/dataflow/temp
```

### What does not change

**The transform graph.** Not one transform is added, removed or
reordered. `ReadFromText`, `ParDo`, `Filter`, `Map`, `CombinePerKey`,
`WriteToText` — the same six steps, the same code, the same labels.

That is the whole argument for the `build_pipeline(pipeline, ...)`
signature. The graph is data, not code that has to be re-written for
each environment. The runner is the only thing that differs.

### The staging and temp locations

Dataflow needs somewhere to put the job's own files — the staged
Python wheel, the serialized graph, and the shuffle scratch space. They
are separate from your output for a reason: Dataflow writes to them
during the run and cleans up after, and a job that shares a temp
location with another job in the same region will corrupt it. The
course's `apply_dataflow_defaults` derives both from a bucket prefix
for exactly that reason.

### The streaming version

Deploying the streaming pipeline is the same, with one addition:

```bash
--streaming \
--temp_location gs://my-bucket/dataflow/temp \
--staging_location gs://my-bucket/dataflow/staging
```

`--streaming` switches the runner into streaming mode, where the job
runs until you cancel it, instead of draining the input and exiting.
A streaming job also needs a Pub/Sub topic instead of a file, and it
costs money while it runs — which is the one place the local runner
and the cloud runner differ in kind rather than in degree.

### The code below

The next cell assembles the Dataflow command. It is **inert** unless
`PROJECT_ID` and `GCS_BUCKET` are set, so the notebook still runs with
no GCP account.

In [ ]:
# --- The deployment command, assembled but not run -------------------------
#
# Everything here is derived from environment variables that are empty
# by default, so this cell does nothing on a machine without GCP
# credentials. It is here so you can see the exact shape of the
# command, not so you can run it.

PROJECT_ID = os.environ.get("PROJECT_ID", "")
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")
REGION = os.environ.get("REGION", "us-central1")

print("PROJECT_ID :", PROJECT_ID or "(not set)")
print("GCS_BUCKET :", GCS_BUCKET or "(not set)")
print("REGION     :", REGION)
print()

if PROJECT_ID and GCS_BUCKET:
    # The GCS paths use the f-string because the bucket and the project
    # are known only at run time.
    input_path = f"gs://{GCS_BUCKET}/orders.csv"
    output_path = f"gs://{GCS_BUCKET}/summary"
    staging = f"gs://{GCS_BUCKET}/dataflow/staging"
    temp = f"gs://{GCS_BUCKET}/dataflow/temp"

    command = [
        sys.executable, str(PROJECT_DIR / "batch_pipeline.py"),
        "--runner", "DataflowRunner",
        "--project", PROJECT_ID,
        "--region", REGION,
        "--input", input_path,
        "--output", output_path,
        "--staging_location", staging,
        "--temp_location", temp,
    ]
    print("The Dataflow command:")
    print()
    print(" \\\n".join(f"   {part}" for part in command))
    print()
    print("Run it with:  subprocess.run(command, check=True)")
else:
    print("PROJECT_ID and GCS_BUCKET are not set, so no Dataflow command is")
    print("assembled. Set them to see the real one:")
    print()
    print("   export PROJECT_ID=my-project")
    print("   export GCS_BUCKET=my-bucket")
    print()
    print("Everything above the `if` still ran — the notebook never needs a")
    print("GCP account. That is the property worth taking away from this")
    print("section: the deployment is a configuration change, not a code")
    print("change, and the local run is a complete rehearsal of it.")

## 🔑 Takeaways

- **The local run is a real rehearsal of the cloud run.** The same
  `build_pipeline` function builds the graph for both. The runner, the
  path prefixes, and four flags are the entire difference.

- **Cross-validate against a second implementation.** `build_revenue_summary`
  is a pure-Python twin of the Beam pipeline. The two share the *rules*
  and not the *mechanism*, which is exactly what makes their agreement
  meaningful. A green run tells you the pipeline did not crash; only the
  comparison tells you it is right.

- **Assert, do not print.** The four expected region totals are
  asserted, so a change that breaks the currency table or moves the
  status filter fails at the point of the change. A number on screen is
  not a check.

- **Idempotency is a design property, not a feature.** Nothing reads the
  wall clock and the output path is an input, so the same input always
  produces the same numbers to a new location. That is what makes a
  backfill safe.

- **The test suite is the durable form of this notebook.** `TestPipeline`
  and `assert_that` are how you keep the guarantees once the code
  changes. Remember the two gotchas: unwrap `WindowedValue`s, and
  `TestPipeline` is not the Direct Runner.

- **Know the difference between the local runner and the cloud one.**
  Everything in this notebook ran with no credentials and no bill. The
  first thing to change when you deploy is not the pipeline — it is
  adding a staging bucket, a temp bucket, and a region.